# VietHandOCR Part 2: Baseline Evaluation

Welcome to **Part 2** of the VietHandOCR pipeline.
- **Previous Notebook**: [Part 1: Data Preparation & EDA](./01_Data_Preparation_and_EDA.ipynb)
- **Next Notebook**: [Part 3: Digital Image Processing (DIP) Pipeline](./03_Digital_Image_Processing.ipynb)

## Introduction
This notebook measures the Zero-shot performance of the pre-trained `vgg_transformer` model on the raw test set. This provides a Baseline to prove the effectiveness of the upcoming DIP steps and Fine-tuning.


In [ ]:
!pip install -q vietocr jiwer nltk Pillow==10.2.0


In [ ]:
import os
import torch
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm
import jiwer
import nltk

nltk.download('punkt', quiet=True)

from vietocr.tool.predictor import Predictor
from vietocr.tool.config import Cfg

def calculate_metrics(predictions, targets):
    '''Calculates CER, WER, Exact Match, and BLEU.'''
    cer = jiwer.cer(targets, predictions)
    wer = jiwer.wer(targets, predictions)
    exact_match = sum(1 for p, t in zip(predictions, targets) if p == t) / len(targets)
    
    bleu_scores = []
    for p, t in zip(predictions, targets):
        reference = [nltk.word_tokenize(t.lower())]
        candidate = nltk.word_tokenize(p.lower())
        if len(candidate) == 0:
            bleu_scores.append(0.0)
            continue
        try:
            bleu = nltk.translate.bleu_score.sentence_bleu(reference, candidate, weights=(0.5, 0.5))
            bleu_scores.append(bleu)
        except Exception:
            bleu_scores.append(0.0)
            
    avg_bleu = sum(bleu_scores) / len(bleu_scores) if bleu_scores else 0.0
    
    return {"CER": cer, "WER": wer, "Exact Match": exact_match, "BLEU": avg_bleu}


In [ ]:
def evaluate_baseline(test_txt_path, images_base_dir, level_name='all'):
    config = Cfg.load_config_from_name('resnet_transformer')
    config['device'] = 'cuda:0' if torch.cuda.is_available() else 'cpu'
    predictor = Predictor(config)
    print("Loaded pre-trained resnet_transformer model successfully!")

    with open(test_txt_path, 'r', encoding='utf-8') as f:
        lines = f.read().strip().split('\n')
        
    targets = []
    predictions = []
    results_detail = []
    
    print(f"  [1/3] ĐỌC DỮ LIỆU ĐẦU VÀO...")
    print(f"    - File input: {test_txt_path}")
    print(f"    - Base Image Dir: {images_base_dir}")
    print(f"    - Tổng số dòng dữ liệu (ảnh): {len(lines)}")
    print(f"  [2/3] BẮT ĐẦU CHẠY SUY LUẬN (INFERENCE)...")
    for line in tqdm(lines):
        if not line.strip(): continue
        parts = line.split('\t')
        if len(parts) != 2: continue
            
        rel_img_path, ground_truth = parts
        full_img_path = os.path.join(images_base_dir, rel_img_path)
        
        try:
            img = Image.open(full_img_path)
            pred = predictor.predict(img)
            
            targets.append(ground_truth)
            predictions.append(pred)
            
            results_detail.append({
                'image_path': rel_img_path,
                'ground_truth': ground_truth,
                'prediction': pred,
                'is_exact_match': ground_truth == pred
            })
        except Exception as e:
            print(f"Error processing {full_img_path}: {e}")
            
    metrics = calculate_metrics(predictions, targets)
    print("\n" + "="*40)
    print(f"🏆 BASELINE RESULTS (Zero-shot) - LEVEL: {level_name.upper()}")
    print("="*40)
    for k, v in metrics.items():
        print(f"{k:<15}: {v:.4f}")
    print("="*40)
    
    df_results = pd.DataFrame(results_detail)
    df_results.to_csv(f'baseline_error_analysis_{level_name}.csv', index=False, encoding='utf-8')
    output_csv = f'baseline_error_analysis_{level_name}.csv'
    print(f"  [3/3] XUẤT KẾT QUẢ ĐÁNH GIÁ...")
    print(f"    - Đường dẫn file output chi tiết: {os.path.join(os.getcwd(), output_csv)}")
    print(f"    - Lưu thành công {len(df_results)} dòng kết quả dự đoán.")
    
    return metrics, df_results


In [ ]:
import os
import glob

# Tìm thư mục dataset
base_dataset_path = 'VietHandOCR_Datasets'
if os.path.exists('../input'):
    for root, dirs, files in os.walk('../input'):
        for d in dirs:
            if 'handocr' in d.lower() and 'data' in d.lower():
                base_dataset_path = os.path.join(root, d)
                break
        if base_dataset_path != 'VietHandOCR_Datasets': break

# Tìm tất cả các file test_*.txt và val_*.txt trong ../input
eval_files = {}
if os.path.exists('../input'):
    for root, dirs, files in os.walk('../input'):
        for file in files:
            if (file.startswith('test_') or file.startswith('val_')) and file.endswith('.txt'):
                # Extract split_level name, e.g. test_line.txt -> test_line
                name = file.replace('.txt', '')
                eval_files[name] = os.path.join(root, file)

if not eval_files:
    print("Warning: Không tìm thấy file test_*.txt hoặc val_*.txt. Vui lòng chạy lại file 01_Data_Preparation_and_EDA.ipynb.")
else:
    print(f"Found dataset at: {base_dataset_path}")
    print(f"Found {len(eval_files)} evaluation files: {list(eval_files.keys())}")
    
    all_metrics = {}
    for name, txt_path in eval_files.items():
        print(f"\n{'='*50}")
        print(f"🚀 EVALUATING: {name.upper()}")
        print(f"{'='*50}")
        
        metrics, df_results = evaluate_baseline(txt_path, base_dataset_path, level_name=name)
        all_metrics[name] = metrics
        
    print(f"\n{'*'*50}")
    print(f"📊 SUMMARY OF ALL EVALUATIONS (Zero-shot ResNet50)")
    print(f"{'*'*50}")
    for name, metrics in all_metrics.items():
        print(f"--- Split: {name.upper()} ---")
        for k, v in metrics.items():
            print(f"  {k:<12}: {v:.4f}")
    print(f"{'*'*50}")
